In [1]:
import os
# --- FIX MEMORIA ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import optuna 
from optuna.trial import TrialState
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import f1_score
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import AutoAugment, AutoAugmentPolicy 
from torch.cuda.amp import autocast, GradScaler

# --- 1. CONFIGURAZIONE ---
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_paths():
    base_train = '/kaggle/input/dataset/archive'
    if not os.path.exists(base_train): base_train = '/kaggle/input/dataset'
    test_img_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.endswith("test_data/images"):
            test_img_path = root; break
    if test_img_path is None:
        test_img_path = '/kaggle/input/dataset/archive (1)/dataset challenge 2 dl /test_data/images'
    return {
        'train_img': os.path.join(base_train, 'images'),
        'train_mask': os.path.join(base_train, 'masks'),
        'labels': os.path.join(base_train, 'train_labels.csv'),
        'test_img': test_img_path, 
        'test_mask': test_img_path.replace('images', 'masks')
    }

PATHS = find_paths()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRECOMPUTED_DIR = '/kaggle/working/train_precomputed_optuna'

# Costanti fisse per il tuning
IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS_PER_TRIAL = 15 # Bastano poche epoche per capire se un set di parametri è buono
SEED = 42

# --- 2. PREPROCESSING & DATASET ---
class ReinhardNormalizer:
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])
    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except: return img_rgb

def get_bbox_from_mask(mask):
    if np.sum(mask) == 0: return None
    rows, cols = np.where(mask > 0)
    return np.min(rows), np.max(rows), np.min(cols), np.max(cols)

def preprocess_and_save(df, src_img, src_mask, out_dir):
    if not os.path.exists(out_dir): os.makedirs(out_dir)
    reinhard = ReinhardNormalizer()
    valid_files = []
    print(f"⚙️ Precomputing images for Optuna -> {out_dir}")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_name = row['sample_index']
        mask_name = img_name.replace("img", "mask")
        img_p = os.path.join(src_img, img_name)
        mask_p = os.path.join(src_mask, mask_name)
        out_p = os.path.join(out_dir, img_name)
        
        if os.path.exists(out_p): valid_files.append(img_name); continue
        try:
            image = cv2.imread(img_p); image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                bbox = get_bbox_from_mask(mask)
                if bbox: image = image[bbox[0]:bbox[1]+1, bbox[2]:bbox[3]+1]
            image = reinhard(image)
            Image.fromarray(image).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS).save(out_p, "PNG")
            valid_files.append(img_name)
        except: continue
    return valid_files

class TissueDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['sample_index'])
        try: image = Image.open(img_path).convert("RGB")
        except: image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform: image = self.transform(image)
        return image, torch.tensor(row['label_encoded'], dtype=torch.long)

def get_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), 
        AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]), transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

# --- 3. MIXUP & MODEL ---
def mixup_data(x, y, alpha=1.0):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    index = torch.randperm(x.size(0)).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def get_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(DEVICE)

# --- 4. OPTUNA OBJECTIVE FUNCTION ---
def objective(trial):
    # 1. Suggerisci Iperparametri
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    mixup_alpha = trial.suggest_float("mixup_alpha", 0.2, 1.0)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.15)
    
    # 2. Setup Dati (Usiamo uno split fisso 80/20 per velocità nel tuning)
    # Nota: Carichiamo i loader qui dentro o li passiamo globali?
    # Per efficienza li usiamo globali (train_loader_tune, val_loader_tune) definiti nel main
    
    # 3. Setup Modello
    model = get_model(num_classes=4) # Assumiamo 4 classi
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_TRIAL)
    scaler = GradScaler()
    
    best_trial_f1 = 0.0
    
    # 4. Training Loop Breve
    for epoch in range(EPOCHS_PER_TRIAL):
        model.train()
        for imgs, lbls in train_loader_tune:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            
            with autocast():
                imgs, ta, tb, lam = mixup_data(imgs, lbls, mixup_alpha)
                out = model(imgs)
                loss = mixup_criterion(criterion, out, ta, tb, lam)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        
        scheduler.step()
        
        # Validation
        model.eval()
        v_preds, v_targets = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader_tune:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                with autocast():
                    out = model(imgs)
                v_preds.extend(out.argmax(1).cpu().numpy())
                v_targets.extend(lbls.cpu().numpy())
        
        val_f1 = f1_score(v_targets, v_preds, average='macro')
        
        # 5. Optuna Pruning (Se va male, interrompi subito)
        trial.report(val_f1, epoch)
        if trial.should_prune():
            # Pulizia prima di uscire
            del model, optimizer, scaler
            torch.cuda.empty_cache()
            gc.collect()
            raise optuna.exceptions.TrialPruned()
            
        if val_f1 > best_trial_f1:
            best_trial_f1 = val_f1
            
    # Pulizia fine trial
    del model, optimizer, scaler
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_trial_f1

# --- MAIN ---
if __name__ == "__main__":
    seed_everything(SEED)
    
    # A. Prepara Dati
    df = pd.read_csv(PATHS['labels'])
    valid_mask = df['sample_index'].apply(lambda x: os.path.exists(os.path.join(PATHS['train_img'], x)))
    df = df[valid_mask].reset_index(drop=True)
    
    le = LabelEncoder()
    df['label_encoded'] = le.fit_transform(df['label'])
    
    # Precomputazione
    valid_train = preprocess_and_save(df, PATHS['train_img'], PATHS['train_mask'], PRECOMPUTED_DIR)
    df_final = df[df['sample_index'].isin(valid_train)].reset_index(drop=True)
    
    # B. Split per Tuning (80/20 Stratified - Veloce)
    # Non usiamo KFold qui perché Optuna deve fare centinaia di trial
    train_idx, val_idx = train_test_split(
        np.arange(len(df_final)), 
        test_size=0.2, 
        stratify=df_final['label_encoded'], 
        random_state=SEED
    )
    
    train_df = df_final.iloc[train_idx].reset_index(drop=True)
    val_df = df_final.iloc[val_idx].reset_index(drop=True)
    
    # C. Dataloaders Globali (per non ricrearli ogni trial)
    counts = train_df['label_encoded'].value_counts().sort_index().values
    weights = 1. / counts
    s_weights = [weights[l] for l in train_df['label_encoded']]
    sampler = WeightedRandomSampler(s_weights, len(s_weights))
    
    train_trans, val_trans = get_transforms()
    
    # Variabili Globali usate dentro objective()
    train_loader_tune = DataLoader(TissueDataset(train_df, PRECOMPUTED_DIR, train_trans), 
                                   batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader_tune = DataLoader(TissueDataset(val_df, PRECOMPUTED_DIR, val_trans), 
                                 batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # D. Optuna Study
    print("\n🔍 Starting Optuna Hyperparameter Search...")
    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))
    # Eseguiamo 25-30 Trial (o per 1-2 ore)
    study.optimize(objective, n_trials=30, timeout=3600*2) 
    
    print("\n🎉 Tuning Completed!")
    print("Best F1:", study.best_value)
    print("Best Params:", study.best_params)
    
    # E. Salva i migliori parametri per uso futuro
    print("\n--- COPIA QUESTI PARAMETRI NEL TUO SCRIPT DI TRAINING FINALE ---")
    for k, v in study.best_params.items():
        print(f"{k.upper()} = {v}")

⚙️ Precomputing images for Optuna -> /kaggle/working/train_precomputed_optuna


100%|██████████| 581/581 [01:13<00:00,  7.95it/s]
[I 2025-12-10 07:58:53,183] A new study created in memory with name: no-name-4b3f17a0-d09a-4756-b32d-fdbcb5a06930
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth



🔍 Starting Optuna Hyperparameter Search...


100%|██████████| 20.5M/20.5M [00:00<00:00, 163MB/s]
/tmp/ipykernel_47/845062906.py:167: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_47/845062906.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_47/845062906.py:195: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_47/845062906.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_47/845062906.py:195: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_47/845062906.py:178: FutureWarning


🎉 Tuning Completed!
Best F1: 0.3840144230769231
Best Params: {'lr': 0.0005139019055168473, 'weight_decay': 0.0011602635806549518, 'mixup_alpha': 0.5757836769023851, 'label_smoothing': 0.118464195337765}

--- COPIA QUESTI PARAMETRI NEL TUO SCRIPT DI TRAINING FINALE ---
LR = 0.0005139019055168473
WEIGHT_DECAY = 0.0011602635806549518
MIXUP_ALPHA = 0.5757836769023851
LABEL_SMOOTHING = 0.118464195337765


Precomputazione per velocizzare il training
Stampe delle epoche di training con train e val loss, val f1
Grafici e confusion matrix 
Creazione submission includendo nel nome la validation f1 
Riproducibilità 



Aumentare numero di epoche e fare early stopping 